# LangGraph version

The same pipeline as a graph: each step is a node, and the graph pauses before the write step until a person approves. The business code is loaded from the cells of `llm_demo.ipynb` tagged `lib`.

In [1]:
# Load the reusable cells (tag "lib") of llm_demo.ipynb into this kernel; their printed output is hidden
import contextlib, io, json, sys, types
import nbformat

source_nb = nbformat.read("llm_demo.ipynb", as_version=4)
lib = types.ModuleType("demo_lib")          # a real module, so that Pydantic can resolve its type hints
sys.modules["demo_lib"] = lib
with contextlib.redirect_stdout(io.StringIO()):
    for cell in source_nb.cells:
        if cell.cell_type == "code" and "lib" in cell.metadata.get("tags", []):
            code = cell.source.replace('TRACE_FILE = "trace.jsonl"', 'TRACE_FILE = "trace_langgraph.jsonl"')   # do not overwrite the other notebook's trace
            exec(compile(code, "llm_demo.ipynb", "exec"), lib.__dict__)
globals().update({k: v for k, v in lib.__dict__.items() if not k.startswith("__")})
print("loaded; primary provider:", PRIMARY, PROVIDERS[PRIMARY]["model"])

loaded; primary provider: openai gpt-4.1-mini


In [2]:
# State of the graph and one function per node (each node only calls code we already have)
from typing import TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph

extractor = LLMExtractor(LLMClient(PRIMARY))
drafter = LLMDrafter(extractor.client)


class FlowState(TypedDict, total=False):
    email_id: str
    text: str
    cargo: dict          # everything in the state is plain JSON, so it can be checkpointed
    report: dict
    action: str
    reason: str
    tms: dict
    draft: str


def n_extract(s: FlowState):
    try:
        cargo = extract_cargo(s["email_id"], s["text"], extractor, trace_id=s["email_id"])
    except NeedsHuman as exc:
        return {"action": "human_review", "reason": str(exc)}
    return {"cargo": cargo.model_dump(mode="json")}


def n_match(s: FlowState):
    return {"report": match_vessels(Cargo.model_validate(s["cargo"]), VESSELS).model_dump(mode="json")}


def n_decide(s: FlowState):
    action, reason = decide(Cargo.model_validate(s["cargo"]), MatchReport.model_validate(s["report"]))
    return {"action": action, "reason": reason}


def n_write(s: FlowState):                                    # the node the graph pauses in front of
    cargo, report = Cargo.model_validate(s["cargo"]), MatchReport.model_validate(s["report"])
    return {"tms": write_to_tms(cargo, report.matches[0], idempotency_key(cargo))}


def n_draft(s: FlowState):
    cargo, report = Cargo.model_validate(s["cargo"]), MatchReport.model_validate(s["report"])
    facts = verified_facts(cargo, report.matches[0])
    text = drafter.draft(facts)
    return {"draft": text + (f"\n[needs review: numbers not in facts {grounding_check(text, facts)}]" if grounding_check(text, facts) else "")}


def after_extract(s: FlowState):
    return "end" if "cargo" not in s else "match"


def after_decide(s: FlowState):
    return "write" if s["action"] == "auto_propose" else "end"     # exceptions never reach the write node


builder = StateGraph(FlowState)
for name, fn in [("extract", n_extract), ("match", n_match), ("decide", n_decide), ("write", n_write), ("draft", n_draft)]:
    builder.add_node(name, fn)
builder.add_edge(START, "extract")
builder.add_conditional_edges("extract", after_extract, {"match": "match", "end": END})
builder.add_edge("match", "decide")
builder.add_conditional_edges("decide", after_decide, {"write": "write", "end": END})
builder.add_edge("write", "draft")
builder.add_edge("draft", END)

# interrupt_before=["write"]: the graph stops here and waits for a person
graph = builder.compile(checkpointer=MemorySaver(), interrupt_before=["write"])
print("nodes:", list(graph.get_graph().nodes))

nodes: ['__start__', 'extract', 'match', 'decide', 'write', 'draft', '__end__']


## Run 1: routine email with approval

Nothing is written while the graph waits.

In [3]:
FAKE_TMS.clear()
config = {"configurable": {"thread_id": "mail-1"}}
state = graph.invoke({"email_id": "mail-1", "text": EMAILS["mail-1"]}, config)
snapshot = graph.get_state(config)
print("paused before:", snapshot.next, "| decision so far:", state["action"], "-", state["reason"])
print("records in the fake TMS while waiting:", len(FAKE_TMS))
assert snapshot.next == ("write",) and not FAKE_TMS

# ... a person looks at the proposal and clicks "approve": resume from the checkpoint
final = graph.invoke(None, config)
print("after approval:", final["tms"])
print(final["draft"])
assert len(FAKE_TMS) == 1

paused before: ('write',) | decision so far: auto_propose - MV Bosphorus Star scores 0.957, clearly ahead
records in the fake TMS while waiting: 0


after approval: {'status': 'created', 'key': '4dd67fa72508'}
Subject: Charter Opportunity for MV Bosphorus Star

Dear Customer,

We are pleased to offer the MV Bosphorus Star for a wheat shipment of 25000 tonnes. The vessel is available within the laycan period from 2026-12-10 to 2026-12-15, with an expected arrival on 2026-12-11. The daily hire rate is USD 8800, and the vessel's utilization is currently at 89.3%.

Please let us know if you are interested.

Best regards,  
Chartering team


## Run 2: no approval, and an exception

If the thread is never resumed, nothing is written. A large cargo never reaches the write node.

In [4]:
FAKE_TMS.clear()
config2 = {"configurable": {"thread_id": "mail-1-rejected"}}
graph.invoke({"email_id": "mail-1", "text": EMAILS["mail-1"]}, config2)
print("waiting for a human at:", graph.get_state(config2).next, "| records in the fake TMS:", len(FAKE_TMS))
assert len(FAKE_TMS) == 0

# an exception (large cargo) never even reaches the write node
config3 = {"configurable": {"thread_id": "mail-2"}}
out = graph.invoke({"email_id": "mail-2", "text": EMAILS["mail-2"]}, config3)
print("mail-2:", out["action"], "-", out["reason"], "| next nodes:", graph.get_state(config3).next)

waiting for a human at: ('write',) | records in the fake TMS: 0


mail-2: human_review - large cargo (60000 t > 40000 t) | next nodes: ()


## Notes

* In-memory checkpoints are lost when the kernel stops; use a database-backed checkpointer for real use.
* For five fixed steps the plain-Python version (`run_mode_a`) is about as short; a graph pays off with loops, branches and long waits.